In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir('/content/drive/MyDrive/greenwashingllmv2/')

Mounted at /content/drive/


In [ ]:
!pip install -q openai pandas tqdm pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("請輸入 OpenAI API Key: ")

請輸入 OpenAI API Key: ··········


In [ ]:
# -*- coding: utf-8 -*-
# =========================================================
# Hybrid Retrieval-Augmented Few-Shot
# ESG / Greenwashing Sentence Scoring with ChatGPT
# FULLY ALIGNED TO LLaMA HYBRID RETRIEVAL FEW-SHOT 7 METRICS
# =========================================================

# =========================================================
# 0. Install (Colab first cell, run once if needed)
# =========================================================
# !pip install -q --upgrade openai pandas tqdm numpy scikit-learn sentence-transformers

# =========================================================
# 1. Imports
# =========================================================
import os
import json
import time
import re
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# =========================================================
# 2. API key
# =========================================================
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("cOPENAI_API_KEY")
if not api_key:
    api_key = getpass("請輸入 OpenAI API Key: ").strip()
    os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

# =========================================================
# 3. Basic settings
# =========================================================
INPUT_FILE = "sentences_for_large_mark.csv"
OUTPUT_FILE = "greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark.csv"

MODEL_NAME = "gpt-5.4"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

SENTENCE_COL = "sentence"

MAX_OUTPUT_TOKENS = 900
USE_FEW_SHOT = True
USE_DYNAMIC_RETRIEVAL = True
RETRY_ON_FAIL = 1
SLEEP_SECONDS = 0.3

# 小測試可打開
# N_TEST_ROWS = 30
N_TEST_ROWS = None

# 固定 few-shot 保留幾個核心例子
N_CORE_FEW_SHOT = 2

# 動態檢索幾個 example
TOP_K_RETRIEVED = 2

EXPECTED_KEYS = [
    "specificity",
    "evidence_substantiation",
    "vagueness",
    "commitment",
    "temporal_credibility",
    "deflection",
    "comparability"
]

SCORE_COLS = [
    "specificity_score",
    "evidence_substantiation_score",
    "vagueness_score",
    "commitment_score",
    "temporal_credibility_score",
    "deflection_score",
    "comparability_score"
]

FALLBACK_JSON = {
    "specificity": {"score": 0, "reason": "fallback"},
    "evidence_substantiation": {"score": 0, "reason": "fallback"},
    "vagueness": {"score": 3, "reason": "fallback"},
    "commitment": {"score": 0, "reason": "fallback"},
    "temporal_credibility": {"score": 0, "reason": "fallback"},
    "deflection": {"score": 0, "reason": "fallback"},
    "comparability": {"score": 0, "reason": "fallback"}
}

# =========================================================
# 4. JSON schema for Structured Outputs
# =========================================================
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "specificity": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "evidence_substantiation": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "vagueness": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "commitment": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "temporal_credibility": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "deflection": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        },
        "comparability": {
            "type": "object",
            "properties": {
                "score": {"type": "integer", "enum": [0, 1, 2, 3, 4, 5]},
                "reason": {"type": "string"}
            },
            "required": ["score", "reason"],
            "additionalProperties": False
        }
    },
    "required": EXPECTED_KEYS,
    "additionalProperties": False
}

# =========================================================
# 5. Load data
# =========================================================
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

print("資料筆數:", len(df))
print("欄位名稱:", df.columns.tolist())

if SENTENCE_COL not in df.columns:
    raise ValueError(f"找不到欄位: {SENTENCE_COL}")

df[SENTENCE_COL] = df[SENTENCE_COL].astype(str).fillna("").str.strip()

if N_TEST_ROWS is not None:
    df = df.head(N_TEST_ROWS).copy()
    print(f"目前只測前 {len(df)} 筆")

# =========================================================
# 6. Load embedding model
# =========================================================
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_ID)
print("Embedding model loaded:", EMBED_MODEL_ID)

# =========================================================
# 7. System prompt
# =========================================================
SYSTEM_PROMPT = """
You are an expert in sustainability communication and greenwashing analysis.

Task:
Evaluate the environmental sentence on exactly seven dimensions from 0 to 5:
1. specificity
2. evidence_substantiation
3. vagueness
4. commitment
5. temporal_credibility
6. deflection
7. comparability

Scoring reminders:
- specificity: higher = more measurable and concrete
- evidence_substantiation: higher = more proof, certification, audit, or supporting data
- vagueness: higher = more vague environmental wording
- commitment: higher = stronger future action commitment
- temporal_credibility: higher = clearer timeline, milestone, or deadline
- deflection: higher = stronger responsibility shifting to consumers, society, partners, or others
- comparability: higher = clearer baseline, benchmark, or comparison reference

STRICT OUTPUT RULES:
- Return ONLY one valid JSON object.
- Do NOT use markdown.
- Do NOT use code fences.
- Do NOT add any explanation before or after JSON.
- Use exactly these seven keys:
  specificity, evidence_substantiation, vagueness, commitment,
  temporal_credibility, deflection, comparability
- Each score must be an integer from 0 to 5.
- Each reason must be under 12 words.
- Start immediately with { and end with }.

Required format:
{
  "specificity":{"score":0,"reason":""},
  "evidence_substantiation":{"score":0,"reason":""},
  "vagueness":{"score":0,"reason":""},
  "commitment":{"score":0,"reason":""},
  "temporal_credibility":{"score":0,"reason":""},
  "deflection":{"score":0,"reason":""},
  "comparability":{"score":0,"reason":""}
}
""".strip()

# =========================================================
# 8. Core few-shot examples
# =========================================================
FEW_SHOT_EXAMPLES = [
    {
        "id": "core_1",
        "sentence": "We care about the planet and are working toward a greener future.",
        "answer": {
            "specificity": {"score": 0, "reason": "No measurable details"},
            "evidence_substantiation": {"score": 0, "reason": "No evidence"},
            "vagueness": {"score": 5, "reason": "Highly vague sentence"},
            "commitment": {"score": 1, "reason": "Weak intention only"},
            "temporal_credibility": {"score": 0, "reason": "No timeline"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline provided"}
        }
    },
    {
        "id": "core_2",
        "sentence": "We aim to reduce water consumption by 25% across factories by 2028 compared with 2020 levels.",
        "answer": {
            "specificity": {"score": 5, "reason": "Clear metric and scope"},
            "evidence_substantiation": {"score": 1, "reason": "No verification method"},
            "vagueness": {"score": 1, "reason": "Mostly concrete statement"},
            "commitment": {"score": 4, "reason": "Clear planned action"},
            "temporal_credibility": {"score": 4, "reason": "Defined deadline"},
            "deflection": {"score": 0, "reason": "Company keeps responsibility"},
            "comparability": {"score": 5, "reason": "Explicit baseline included"}
        }
    },
    {
        "id": "core_3",
        "sentence": "Our packaging is more sustainable and better for the environment.",
        "answer": {
            "specificity": {"score": 0, "reason": "No material details"},
            "evidence_substantiation": {"score": 0, "reason": "No supporting evidence"},
            "vagueness": {"score": 5, "reason": "Uses undefined terms"},
            "commitment": {"score": 1, "reason": "No concrete action"},
            "temporal_credibility": {"score": 0, "reason": "No timeframe stated"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No comparison baseline"}
        }
    },
    {
        "id": "core_4",
        "sentence": "By 2030, all cotton in our products will be certified organic or recycled.",
        "answer": {
            "specificity": {"score": 4, "reason": "Specific material deadline"},
            "evidence_substantiation": {"score": 1, "reason": "No proof mechanism"},
            "vagueness": {"score": 1, "reason": "Sentence is concrete"},
            "commitment": {"score": 5, "reason": "Strong future commitment"},
            "temporal_credibility": {"score": 5, "reason": "Clear end date"},
            "deflection": {"score": 0, "reason": "Company retains responsibility"},
            "comparability": {"score": 0, "reason": "No baseline comparison"}
        }
    }
]

# =========================================================
# 9. Example bank for retrieval
# =========================================================
EXAMPLE_BANK = [
    {
        "id": "bank_1",
        "sentence": "Our operations are environmentally friendly and sustainable.",
        "answer": {
            "specificity": {"score": 0, "reason": "No measurable information"},
            "evidence_substantiation": {"score": 0, "reason": "No supporting proof"},
            "vagueness": {"score": 5, "reason": "Highly vague wording"},
            "commitment": {"score": 1, "reason": "Weak environmental intention"},
            "temporal_credibility": {"score": 0, "reason": "No timeframe"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline given"}
        }
    },
    {
        "id": "bank_2",
        "sentence": "We will cut Scope 1 emissions by 40% by 2030.",
        "answer": {
            "specificity": {"score": 5, "reason": "Clear metric and scope"},
            "evidence_substantiation": {"score": 1, "reason": "No verification specified"},
            "vagueness": {"score": 0, "reason": "Concrete statement"},
            "commitment": {"score": 5, "reason": "Strong commitment"},
            "temporal_credibility": {"score": 5, "reason": "Clear deadline"},
            "deflection": {"score": 0, "reason": "Company owns responsibility"},
            "comparability": {"score": 0, "reason": "No baseline stated"}
        }
    },
    {
        "id": "bank_3",
        "sentence": "We support greener logistics and better packaging solutions.",
        "answer": {
            "specificity": {"score": 1, "reason": "Very limited detail"},
            "evidence_substantiation": {"score": 0, "reason": "No evidence provided"},
            "vagueness": {"score": 4, "reason": "Broad positive wording"},
            "commitment": {"score": 2, "reason": "Weak intended action"},
            "temporal_credibility": {"score": 0, "reason": "No stated timeline"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline reference"}
        }
    },
    {
        "id": "bank_4",
        "sentence": "All manufacturing sites will use 100% renewable electricity by 2027.",
        "answer": {
            "specificity": {"score": 5, "reason": "Specific target and scope"},
            "evidence_substantiation": {"score": 1, "reason": "No validation named"},
            "vagueness": {"score": 0, "reason": "Highly concrete sentence"},
            "commitment": {"score": 5, "reason": "Explicit future action"},
            "temporal_credibility": {"score": 5, "reason": "Clear target year"},
            "deflection": {"score": 0, "reason": "Company owns action"},
            "comparability": {"score": 0, "reason": "No comparison baseline"}
        }
    },
    {
        "id": "bank_5",
        "sentence": "Our products are eco-friendly and designed for a better future.",
        "answer": {
            "specificity": {"score": 0, "reason": "No concrete product detail"},
            "evidence_substantiation": {"score": 0, "reason": "No proof shown"},
            "vagueness": {"score": 5, "reason": "Undefined green terms"},
            "commitment": {"score": 1, "reason": "No strong commitment"},
            "temporal_credibility": {"score": 0, "reason": "No time reference"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline given"}
        }
    },
    {
        "id": "bank_6",
        "sentence": "We plan to reduce plastic packaging by 15% across Europe by 2026.",
        "answer": {
            "specificity": {"score": 4, "reason": "Metric scope and action"},
            "evidence_substantiation": {"score": 1, "reason": "No substantiation given"},
            "vagueness": {"score": 1, "reason": "Mostly precise wording"},
            "commitment": {"score": 4, "reason": "Strong stated plan"},
            "temporal_credibility": {"score": 4, "reason": "Defined deadline"},
            "deflection": {"score": 0, "reason": "Company keeps responsibility"},
            "comparability": {"score": 0, "reason": "No baseline provided"}
        }
    },
    {
        "id": "bank_7",
        "sentence": "Certified by FSC for all paper-based packaging materials.",
        "answer": {
            "specificity": {"score": 3, "reason": "Some scope is defined"},
            "evidence_substantiation": {"score": 5, "reason": "Explicit certification stated"},
            "vagueness": {"score": 1, "reason": "Limited vague wording"},
            "commitment": {"score": 1, "reason": "Present status only"},
            "temporal_credibility": {"score": 0, "reason": "No timeline mentioned"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline comparison"}
        }
    },
    {
        "id": "bank_8",
        "sentence": "We aim to achieve net zero emissions in the long term.",
        "answer": {
            "specificity": {"score": 1, "reason": "Target lacks detail"},
            "evidence_substantiation": {"score": 0, "reason": "No methodology included"},
            "vagueness": {"score": 4, "reason": "Long term is vague"},
            "commitment": {"score": 3, "reason": "Moderate intention stated"},
            "temporal_credibility": {"score": 1, "reason": "No clear deadline"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline given"}
        }
    },
    {
        "id": "bank_9",
        "sentence": "Third-party audited carbon data will be published annually starting in 2026.",
        "answer": {
            "specificity": {"score": 4, "reason": "Clear reporting action"},
            "evidence_substantiation": {"score": 4, "reason": "Third-party audit mentioned"},
            "vagueness": {"score": 1, "reason": "Mostly precise wording"},
            "commitment": {"score": 4, "reason": "Clear future action"},
            "temporal_credibility": {"score": 4, "reason": "Start year specified"},
            "deflection": {"score": 0, "reason": "Company owns reporting"},
            "comparability": {"score": 0, "reason": "No comparison baseline"}
        }
    },
    {
        "id": "bank_10",
        "sentence": "We are committed to making our business more sustainable.",
        "answer": {
            "specificity": {"score": 0, "reason": "No measurable commitment"},
            "evidence_substantiation": {"score": 0, "reason": "No supporting basis"},
            "vagueness": {"score": 5, "reason": "Very broad wording"},
            "commitment": {"score": 2, "reason": "General commitment only"},
            "temporal_credibility": {"score": 0, "reason": "No timing given"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 0, "reason": "No baseline reference"}
        }
    },
    {
        "id": "bank_11",
        "sentence": "Consumers can help reduce plastic waste by recycling our bottles.",
        "answer": {
            "specificity": {"score": 2, "reason": "Some action mentioned"},
            "evidence_substantiation": {"score": 0, "reason": "No evidence given"},
            "vagueness": {"score": 3, "reason": "Limited operational detail"},
            "commitment": {"score": 1, "reason": "Weak company commitment"},
            "temporal_credibility": {"score": 0, "reason": "No timeline stated"},
            "deflection": {"score": 5, "reason": "Shifts responsibility consumers"},
            "comparability": {"score": 0, "reason": "No baseline provided"}
        }
    },
    {
        "id": "bank_12",
        "sentence": "We reduced emissions by 25% compared with 2020 levels.",
        "answer": {
            "specificity": {"score": 5, "reason": "Precise quantified reduction"},
            "evidence_substantiation": {"score": 1, "reason": "No verification cited"},
            "vagueness": {"score": 0, "reason": "Very concrete wording"},
            "commitment": {"score": 1, "reason": "Reports outcome only"},
            "temporal_credibility": {"score": 2, "reason": "Reference year provided"},
            "deflection": {"score": 0, "reason": "No responsibility shifting"},
            "comparability": {"score": 5, "reason": "Explicit baseline comparison"}
        }
    }
]

# =========================================================
# 10. Prepare core few-shot examples
# =========================================================
CORE_FEW_SHOT_EXAMPLES = FEW_SHOT_EXAMPLES[:N_CORE_FEW_SHOT]
core_ids = set(ex["id"] for ex in CORE_FEW_SHOT_EXAMPLES)

# 避免 dynamic retrieval 把 core example 自己又抓進來
RETRIEVAL_BANK = [ex for ex in EXAMPLE_BANK if ex["id"] not in core_ids]

# =========================================================
# 11. Encode retrieval bank
# =========================================================
print("Encoding example retrieval bank...")

retrieval_texts = [x["sentence"] for x in RETRIEVAL_BANK]
retrieval_embeddings = embed_model.encode(
    retrieval_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Retrieval bank encoded:", len(RETRIEVAL_BANK), "examples")

# =========================================================
# 12. Helpers
# =========================================================
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def retrieve_similar_examples(sentence: str, top_k: int = 2):
    """
    Retrieve top-k similar examples from the retrieval bank.
    """
    sentence = normalize_text(sentence)

    if len(RETRIEVAL_BANK) == 0:
        return []

    sentence_embedding = embed_model.encode(
        [sentence],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    sims = cosine_similarity(sentence_embedding, retrieval_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]

    retrieved = []
    for idx in top_indices:
        ex = RETRIEVAL_BANK[int(idx)].copy()
        ex["similarity"] = float(sims[idx])
        retrieved.append(ex)

    return retrieved

# =========================================================
# 13. Build messages
# =========================================================
def make_msg(role: str, text: str):
    return {"role": role, "content": text}

def build_messages(sentence: str):
    sentence = normalize_text(sentence)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    retrieved_examples = []

    # 固定核心 few-shot
    if USE_FEW_SHOT:
        for ex in CORE_FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": f"Sentence: {ex['sentence']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps(ex["answer"], ensure_ascii=False)
            })

    # 動態 retrieval examples
    if USE_DYNAMIC_RETRIEVAL:
        retrieved_examples = retrieve_similar_examples(sentence, top_k=TOP_K_RETRIEVED)

        for ex in retrieved_examples:
            messages.append({"role": "user", "content": f"Sentence: {ex['sentence']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps(ex["answer"], ensure_ascii=False)
            })

    # target sentence
    messages.append({"role": "user", "content": f"Sentence: {sentence}"})

    return messages, retrieved_examples

def build_repair_messages(sentence: str):
    sentence = normalize_text(sentence)

    repair_system_prompt = """
You are a strict JSON generator.

Evaluate the environmental sentence on exactly seven dimensions from 0 to 5:
1. specificity
2. evidence_substantiation
3. vagueness
4. commitment
5. temporal_credibility
6. deflection
7. comparability

Scoring reminders:
- specificity: higher = more measurable and concrete
- evidence_substantiation: higher = more proof, certification, audit, or supporting data
- vagueness: higher = more vague environmental wording
- commitment: higher = stronger future action commitment
- temporal_credibility: higher = clearer timeline, milestone, or deadline
- deflection: higher = stronger responsibility shifting to others
- comparability: higher = clearer baseline or comparison

Return ONLY one valid JSON object.
Do not add any extra text.
Do not use markdown.
Do not use code fences.
Scores must be integers from 0 to 5.
Reasons must be very short, under 8 words.

Required format:
{
"specificity":{"score":0,"reason":""},
"evidence_substantiation":{"score":0,"reason":""},
"vagueness":{"score":0,"reason":""},
"commitment":{"score":0,"reason":""},
"temporal_credibility":{"score":0,"reason":""},
"deflection":{"score":0,"reason":""},
"comparability":{"score":0,"reason":""}
}
""".strip()

    return [
        {"role": "system", "content": repair_system_prompt},
        {"role": "user", "content": f"Sentence: {sentence}"}
    ]

# =========================================================
# 14. Validation helpers
# =========================================================
def normalize_parsed_json(obj):
    if not isinstance(obj, dict):
        return obj

    for key in EXPECTED_KEYS:
        if key in obj and isinstance(obj[key], dict):
            score = obj[key].get("score", None)
            reason = obj[key].get("reason", "")

            if isinstance(score, str):
                score_str = score.strip()
                if score_str.isdigit():
                    obj[key]["score"] = int(score_str)

            if reason is None:
                obj[key]["reason"] = ""
            else:
                obj[key]["reason"] = str(reason)

    return obj

def validate_parsed_json(obj):
    if not isinstance(obj, dict):
        return False

    for key in EXPECTED_KEYS:
        if key not in obj:
            return False

        val = obj[key]
        if not isinstance(val, dict):
            return False

        if "score" not in val or "reason" not in val:
            return False

        score = val["score"]
        reason = val["reason"]

        if not isinstance(score, int):
            return False
        if score < 0 or score > 5:
            return False
        if not isinstance(reason, str):
            return False

    return True

# =========================================================
# 15. Response parsing
# =========================================================
def try_parse_json(text: str):
    if not text:
        return None

    try:
        obj = json.loads(text)
    except Exception:
        return None

    obj = normalize_parsed_json(obj)
    if validate_parsed_json(obj):
        return obj
    return None

def extract_text_from_response_output(response):
    texts = []

    try:
        for item in response.output:
            if getattr(item, "type", None) == "message":
                for c in getattr(item, "content", []):
                    text_val = getattr(c, "text", None)
                    if text_val:
                        texts.append(str(text_val).strip())
    except Exception:
        pass

    return texts

# =========================================================
# 16. Scoring call
# =========================================================
def score_sentence(messages):
    response = client.responses.create(
        model=MODEL_NAME,
        temperature=0,
        input=messages,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        text={
            "format": {
                "type": "json_schema",
                "name": "greenwashing_scores_7metrics",
                "strict": True,
                "schema": JSON_SCHEMA
            }
        }
    )

    raw_text = ""
    parsed = None
    parse_source = ""

    try:
        raw_text = response.output_text.strip()
    except Exception:
        raw_text = ""

    parsed = try_parse_json(raw_text)
    if parsed is not None:
        parse_source = "response.output_text"
        return raw_text, parsed, parse_source

    nested_texts = extract_text_from_response_output(response)
    for nt in nested_texts:
        parsed = try_parse_json(nt)
        if parsed is not None:
            raw_text = nt
            parse_source = "response.output[*].content[*].text"
            return raw_text, parsed, parse_source

    return raw_text, None, "unparsed"

# =========================================================
# 17. Main scoring loop
# =========================================================
results = []
start_time = time.time()

for i, row in tqdm(df.iterrows(), total=len(df), desc="Scoring ChatGPT retrieval few-shot 7 metrics"):
    sentence = str(row[SENTENCE_COL]).strip()

    messages, retrieved_examples = build_messages(sentence)
    raw, parsed, parse_source = score_sentence(messages)

    retry_count = 0
    while parsed is None and retry_count < RETRY_ON_FAIL:
        time.sleep(SLEEP_SECONDS)
        raw, parsed, parse_source = score_sentence(messages)
        retry_count += 1

    if parsed is None:
        repair_messages = build_repair_messages(sentence)
        time.sleep(SLEEP_SECONDS)
        raw, parsed, parse_source = score_sentence(repair_messages)

    print(f"RAW {i+1}: {raw[:180] if raw else '[EMPTY]'}")

    row_dict = row.to_dict()
    row_dict["model_name"] = MODEL_NAME
    row_dict["embedding_model_name"] = EMBED_MODEL_ID
    row_dict["raw_llm_output"] = raw
    row_dict["prompt_type"] = "hybrid_retrieval_augmented_fewshot_7metrics"
    row_dict["parse_source"] = parse_source
    row_dict["retry_on_fail"] = RETRY_ON_FAIL
    row_dict["n_retrieved_examples"] = len(retrieved_examples)
    row_dict["full_prompt_setting"] = f"core{N_CORE_FEW_SHOT}_retrieved{TOP_K_RETRIEVED}"
    row_dict["core_example_ids"] = "|".join([x["id"] for x in CORE_FEW_SHOT_EXAMPLES])

    if len(retrieved_examples) > 0:
        row_dict["retrieved_example_ids"] = "|".join([x["id"] for x in retrieved_examples])
        row_dict["retrieved_example_sentences"] = " ||| ".join([x["sentence"] for x in retrieved_examples])
        row_dict["retrieved_example_similarities"] = "|".join([f"{x['similarity']:.4f}" for x in retrieved_examples])
    else:
        row_dict["retrieved_example_ids"] = ""
        row_dict["retrieved_example_sentences"] = ""
        row_dict["retrieved_example_similarities"] = ""

    if parsed is None:
        parsed = FALLBACK_JSON
        row_dict["json_parse_success"] = False
        row_dict["used_fallback"] = True
    else:
        row_dict["json_parse_success"] = True
        row_dict["used_fallback"] = False

    for key in EXPECTED_KEYS:
        row_dict[f"{key}_score"] = parsed[key]["score"]
        row_dict[f"{key}_reason"] = parsed[key]["reason"]

    results.append(row_dict)
    time.sleep(SLEEP_SECONDS)

elapsed = time.time() - start_time

# =========================================================
# 18. Save results
# =========================================================
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_FILE)

# =========================================================
# 19. Basic stats
# =========================================================
print("\nJSON success:")
print(df_out["json_parse_success"].value_counts(dropna=False))

print("\nUsed fallback:")
print(df_out["used_fallback"].value_counts(dropna=False))

print("\nParse source:")
if "parse_source" in df_out.columns:
    print(df_out["parse_source"].value_counts(dropna=False))

print("\nMissing values:")
print(df_out[SCORE_COLS].isna().sum())

if "label" in df_out.columns:
    print("\nLabel mean:")
    grp = df_out.groupby("label")[SCORE_COLS].mean()
    print(grp)

    if 0 in grp.index and 1 in grp.index:
        gap = grp.loc[0] - grp.loc[1]
        print("\nLabel gap (label 0 - label 1):")
        print(gap)

        print("\nDirection check (expected signs):")
        expected_sign = {
            "specificity_score": "+",
            "evidence_substantiation_score": "+",
            "vagueness_score": "-",
            "commitment_score": "+",
            "temporal_credibility_score": "+",
            "deflection_score": "-",
            "comparability_score": "+"
        }

        for col in SCORE_COLS:
            val = gap[col]
            actual = "+" if val > 0 else "-" if val < 0 else "0"
            print(f"{col}: gap={val:.6f}, expected={expected_sign[col]}, actual={actual}")

# =========================================================
# 20. Save fallback rows separately
# =========================================================
fallback_df = df_out[df_out["used_fallback"] == True].copy()
if len(fallback_df) > 0:
    fallback_file = OUTPUT_FILE.replace(".csv", "_fallback_rows.csv")
    fallback_df.to_csv(fallback_file, index=False, encoding="utf-8-sig")
    print("\nFallback rows saved:", fallback_file)

# =========================================================
# 21. Save retrieval summary
# =========================================================
summary_rows = []

summary_rows.append({"metric": "n_rows", "value": len(df_out)})
summary_rows.append({"metric": "elapsed_seconds", "value": round(elapsed, 2)})
summary_rows.append({"metric": "elapsed_minutes", "value": round(elapsed / 60, 2)})
summary_rows.append({"metric": "json_success_rate", "value": round(df_out["json_parse_success"].mean(), 6)})
summary_rows.append({"metric": "fallback_rate", "value": round(df_out["used_fallback"].mean(), 6)})
summary_rows.append({"metric": "n_core_fewshot", "value": N_CORE_FEW_SHOT})
summary_rows.append({"metric": "n_dynamic_retrieved", "value": TOP_K_RETRIEVED})
summary_rows.append({"metric": "retrieval_bank_size", "value": len(RETRIEVAL_BANK)})
summary_rows.append({"metric": "model_name", "value": MODEL_NAME})
summary_rows.append({"metric": "embedding_model_name", "value": EMBED_MODEL_ID})

if "parse_source" in df_out.columns:
    for k, v in df_out["parse_source"].value_counts(dropna=False).to_dict().items():
        summary_rows.append({"metric": f"parse_source_{k}", "value": v})

if "label" in df_out.columns:
    grp = df_out.groupby("label")[SCORE_COLS].mean()
    if 0 in grp.index and 1 in grp.index:
        gap = grp.loc[0] - grp.loc[1]
        for col in SCORE_COLS:
            summary_rows.append({
                "metric": f"label_gap_{col}",
                "value": round(float(gap[col]), 6)
            })

summary_df = pd.DataFrame(summary_rows)
summary_file = OUTPUT_FILE.replace(".csv", "_summary.csv")
summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
print("Summary saved:", summary_file)

print(f"\nTotal time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
print("\nDone.")

資料筆數: 472
欄位名稱: ['sentence_id', 'folder_year', 'sic', 'company', 'ticker', 'report_year', 'source_folder', 'file', 'rank', 'score', 'page', 'section_guess', 'topic_guess', 'sentence_type', 'has_number', 'has_year', 'has_by_year', 'has_percent', 'has_scope', 'has_sbti', 'has_netzero', 'has_kpi', 'has_material', 'has_green_marketing', 'sentence']
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

RAW 471: {"specificity":{"score":5,"reason":"Quantified target, scopes, baseline, deadline stated"},"evidence_substantiation":{"score":1,"reason":"No supporting evidence or verification men


Scoring ChatGPT retrieval few-shot 7 metrics: 100%|█████████▉| 471/472 [24:46<00:03,  3.14s/it]

RAW 472: {"specificity":{"score":2,"reason":"Some supplier figures appear, but sentence is fragmented"},"evidence_substantiation":{"score":1,"reason":"Numbers shown, but unsupported and unc


Scoring ChatGPT retrieval few-shot 7 metrics: 100%|██████████| 472/472 [24:49<00:00,  3.16s/it]

Saved: greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark.csv

JSON success:
json_parse_success
True    472
Name: count, dtype: int64

Used fallback:
used_fallback
False    472
Name: count, dtype: int64

Parse source:
parse_source
response.output_text    472
Name: count, dtype: int64

Missing values:
specificity_score                0
evidence_substantiation_score    0
vagueness_score                  0
commitment_score                 0
temporal_credibility_score       0
deflection_score                 0
comparability_score              0
dtype: int64
Summary saved: greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored_large_mark_summary.csv

Total time: 1489.70 seconds (24.83 minutes)

Done.


In [ ]:
#哪些 retrieval examples 最常被抓到

In [ ]:
import pandas as pd

df = pd.read_csv("greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv")

# =====================================================
# 1 Retrieval frequency
# =====================================================

ids = []

for x in df["retrieved_example_ids"]:
    if pd.isna(x) or x == "":
        continue
    ids.extend(x.split("|"))

freq_df = pd.Series(ids).value_counts().reset_index()
freq_df.columns = ["retrieved_id", "count"]

freq_df.to_csv("retrieval_frequency.csv", index=False, encoding="utf-8-sig")

# =====================================================
# 2 Similarity stats
# =====================================================

sim_means = []
sim_maxs = []
sim_mins = []

for x in df["retrieved_example_similarities"]:
    if pd.isna(x) or x == "":
        continue

    sims = [float(i) for i in x.split("|")]

    sim_means.append(sum(sims) / len(sims))
    sim_maxs.append(max(sims))
    sim_mins.append(min(sims))

sim_df = pd.DataFrame({
    "sim_mean": sim_means,
    "sim_max": sim_maxs,
    "sim_min": sim_mins
})

sim_stats = sim_df.describe()

sim_stats.to_csv("retrieval_similarity_stats.csv", encoding="utf-8-sig")

# =====================================================
# 3 Retrieved pair frequency
# =====================================================

pairs = []

for x in df["retrieved_example_ids"]:
    if pd.isna(x) or x == "":
        continue

    parts = x.split("|")

    if len(parts) == 2:
        pairs.append("|".join(parts))

pair_df = pd.Series(pairs).value_counts().reset_index()
pair_df.columns = ["retrieved_pair", "count"]

pair_df.to_csv("retrieved_pair_frequency.csv", index=False, encoding="utf-8-sig")

print("CSV files saved.")

CSV files saved.


In [ ]:
#chaatgpt7指標 zeroshot fewshot retrieval shot

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# ====== 1. 檔案路徑 ======
files = {
    "Zero-shot": "greenclaims_groundtruth_chatgpt_zeroshot_7metrics_scored_aligned.csv",
    "Few-shot": "greenclaims_groundtruth_chatgpt_fewshot_7metrics_fully_aligned_to_llama.csv",
    "Hybrid": "greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv"
}

# ====== 2. 指標欄位 ======
metrics = [
    "specificity_score",
    "evidence_substantiation_score",
    "vagueness_score",
    "commitment_score",
    "temporal_credibility_score",
      "deflection_score",
    "comparability_score"
]

# ====== 3. Cohen's d ======
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled = np.sqrt(((nx-1)*vx + (ny-1)*vy) / (nx+ny-2))
    if pooled == 0:
        return np.nan
    return (np.mean(x) - np.mean(y)) / pooled

# ====== 4. 計算 ======
rows = []

for method, path in files.items():

    df = pd.read_csv(path)

    for m in metrics:

        g0 = df[df["label"] == 0][m]
        g1 = df[df["label"] == 1][m]

        mean_diff = g0.mean() - g1.mean()
        d = cohens_d(g0, g1)

        try:
            auc = roc_auc_score(df["label"], df[m])
            auc = max(auc, 1-auc)   # 修正方向
        except:
            auc = np.nan

        rows.append({
            "Metric": m.replace("_score","").capitalize(),
            "Method": method,
            "MeanDiff": round(mean_diff,3),
            "Cohen_d": round(d,3),
            "AUC": round(auc,3)
        })

res = pd.DataFrame(rows)

# ====== 5. Pivot reviewer table ======
table = res.pivot(index="Metric", columns="Method")

# 排序欄位
table = table.reindex(
    columns=pd.MultiIndex.from_product(
        [["MeanDiff","Cohen_d","AUC"],["Zero-shot","Few-shot","Hybrid"]]
    )
)

# ====== 6. 輸出 ======
print("\n=== chatgpt7-Metric Prompt Strategy Comparison ===\n")
print(table)

table.to_csv("chatgpt7metrics_prompt_strategy_comparison_table.csv")


=== chatgpt7-Metric Prompt Strategy Comparison ===

                         MeanDiff                   Cohen_d                  \
                        Zero-shot Few-shot Hybrid Zero-shot Few-shot Hybrid   
Metric                                                                        
Commitment                  0.941    0.874  0.710     0.914    0.813  0.579   
Comparability               0.042    0.087  0.029     0.040    0.084  0.028   
Deflection                  0.030    0.023 -0.156     0.020    0.016 -0.089   
Evidence_substantiation     0.306    0.278  0.374     0.458    0.397  0.486   
Specificity                 0.178    0.370  0.339     0.147    0.298  0.281   
Temporal_credibility        0.659    0.681  0.558     0.441    0.503  0.454   
Vagueness                  -0.061   -0.002  0.050    -0.053   -0.002  0.040   

                              AUC                  
                        Zero-shot Few-shot Hybrid  
Metric                                             
